# 01 선생 생성 (Qwen3.8-27B, vLLM)

| 항목 | 값 |
|---|---|
| GPU | A100 40GB → `Qwen3.8-27B-FP8`, A100 80GB → `Qwen3.8-27B` bf16 (자동 선택) |
| 예상 시간 | 약 1시간 10분 (다운로드 15~20분, 시범 50개 2분 + 확인, 학습·검증 3,200개와 재시도 20~30분, 평가 934개와 시간 측정 15분) |
| 예상 컴퓨트 | 약 13~15 CU (A100 약 13 CU/시간 가정. Colab 리소스 창에서 실제 소모율을 확인해 기록) |
| 디스크 | 가중치 29GB(FP8) 또는 56GB(bf16) |
| 이어 하기 | 출력은 256개마다 Drive `teacher/*.jsonl`에 이어 쓴다. 끊기면 처음부터 다시 실행하면 이미 있는 id는 건너뛴다. |

사전 등록: `docs/spec.md` §6.1~6.3, `docs/decisions.md` 2026-11-10. 설정을 바꾸면 사전 등록 위반이니 멈추고 기록한다.
준비: repo에서 `scripts/pack_distill_bundle.sh`로 만든 `tomato_distill_bundle.tar.gz`를 Drive `MyDrive/tomato-distill/`에 올린다.

In [ ]:
# 공통 준비: Drive 연결, 묶음 풀기, 공용 코드 import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, time, gc
DIR = "/content/drive/MyDrive/tomato-distill"
os.environ["TOMATO_DISTILL_DIR"] = DIR
if not os.path.exists(f"{DIR}/bundle/notebooks/distill_common.py"):
    os.makedirs(f"{DIR}/bundle", exist_ok=True)
    !tar xzf {DIR}/tomato_distill_bundle.tar.gz -C {DIR}/bundle
sys.path.insert(0, f"{DIR}/bundle/notebooks")
import distill_common as dc
prompt = dc.setup()
from tomato.explain import grounding
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U vllm
import vllm, torch
print("vllm", vllm.__version__, "torch", torch.__version__)

## 선생 로드 (GPU 메모리로 정밀도 결정, revision 고정, 사고 출력 끔)

In [ ]:
from vllm import LLM, SamplingParams
GB = dc.gpu_gb()
TID, TREV = dc.TEACHERS[80 if GB >= 70 else 40]
llm = LLM(model=TID, revision=TREV, max_model_len=4096, gpu_memory_utilization=0.90,
          limit_mm_per_prompt={"image": 0, "video": 0}, seed=dc.SEED)
CT = {"enable_thinking": False}
meta = {"teacher": TID, "revision": TREV, "gpu": torch.cuda.get_device_name(0), "gpu_gb": GB,
        "vllm": vllm.__version__, "prompt_version": prompt.PROMPT_VERSION, "sampling_train": dc.TEACHER_SAMPLING}
print(meta)

## 생성 함수

In [ ]:
def id_seed(rid, attempt):
    return int(rid.split("-", 1)[1].replace("-", "")) + attempt * 100_000_000

def generate(recs, path, attempt, sampling, step=256):
    done = dc.done_ids(path)
    todo = [r for r in recs if r["id"] not in done]
    for k in range(0, len(todo), step):
        chunk = todo[k:k + step]
        sps = [SamplingParams(**sampling, seed=id_seed(r["id"], attempt)) for r in chunk]
        t0 = time.time()
        outs = llm.chat([prompt.messages(r["input"]) for r in chunk], sps, chat_template_kwargs=CT, use_tqdm=False)
        sec = time.time() - t0
        rows = []
        for r, o in zip(chunk, outs):
            text = o.outputs[0].text.strip()
            rows.append({"id": r["id"], "split": r["split"], "attempt": attempt, "text": text,
                         "batch_sec": round(sec, 2), **grounding.check(text, r["input"])})
        dc.append_jsonl(path, rows)
        print(f"{path.name} {len(done) + k + len(chunk)}/{len(recs)} ({sec:.0f}초)", flush=True)

def ok(row):
    return row["pass"] and row["format"]

RULE_KEYS = ("pass", "format", "a", "a_numbers", "a_median", "b", "c", "c_caution", "perf_caution")
T = dc.DRIVE / "teacher"

## 시범 생성 (spec §6.1): 학습용 입력 50개(seed 20261113), 첫 시도만, 학습 설정의 샘플링
- 첫 시도 통과율(전체·규칙별)과 실패 원문을 보고, 프롬프트를 고칠지 사용자가 정한다.
- 고치면 repo에서 `PROMPT_VERSION`을 올리고 decisions에 기록한 뒤, 묶음을 다시 만들어 올리고 이 노트북을 처음부터 다시 실행한다.
- 시범 출력은 학습에 쓰지 않는다.

In [ ]:
pilot = dc.pilot_subset()
ppath = T / f"pilot_{prompt.PROMPT_VERSION}.jsonl"
generate(pilot, ppath, 0, dc.TEACHER_SAMPLING)
prows = dc.read_jsonl(ppath)
meta["pilot"] = {"prompt_version": prompt.PROMPT_VERSION, "n": len(prows), **{k: sum(r[k] for r in prows) for k in RULE_KEYS}}
print(json.dumps(meta["pilot"], ensure_ascii=False))
for r in [r for r in prows if not ok(r)][:5]:
    print("----", r["id"], r["errors"]); print(r["text"])

## 프롬프트 고정: 시범 결과를 보고 고정할 버전을 적는다. 이 셀부터가 전체 생성이다

In [ ]:
FREEZE = None  # 시범 결과를 본 뒤 고정할 PROMPT_VERSION(예: "v2")을 넣는다
assert FREEZE == prompt.PROMPT_VERSION, f"시범을 확인하고 FREEZE = {prompt.PROMPT_VERSION!r}로 고정한다"
assert (T / f"pilot_{FREEZE}.jsonl").exists(), "이 버전의 시범 생성이 없다"
meta["prompt_frozen"] = {"version": FREEZE, "at": time.strftime("%Y-%m-%d %H:%M:%S")}
print(meta["prompt_frozen"])

## 학습(2003~2021) 3,000개 + 검증(2022) 200개 생성, 근거 검사 실패는 seed를 바꿔 최대 2번 재생성

In [ ]:
train, val = dc.subsets()
for name, recs in (("train", train), ("val", val)):
    remaining = recs
    for attempt in range(dc.MAX_RETRIES + 1):
        path = T / f"{name}_a{attempt}.jsonl"
        generate(remaining, path, attempt, dc.TEACHER_SAMPLING)
        passed = {r["id"] for a in range(attempt + 1) for r in dc.read_jsonl(T / f"{name}_a{a}.jsonl") if ok(r)}
        remaining = [r for r in recs if r["id"] not in passed]
        print(f"{name} attempt {attempt}: 통과 누적 {len(passed)}/{len(recs)}")
        if not remaining:
            break

In [ ]:
# 최종 학습 데이터: id마다 처음 통과한 출력. 끝까지 실패한 id는 뺀다.
summary = {}
for name, recs in (("train", train), ("val", val)):
    best = {}
    for a in range(dc.MAX_RETRIES + 1):
        for r in dc.read_jsonl(T / f"{name}_a{a}.jsonl"):
            if ok(r) and r["id"] not in best:
                best[r["id"]] = r
    final = T / f"{name}_final.jsonl"
    final.unlink(missing_ok=True)
    dc.append_jsonl(final, [best[r["id"]] for r in recs if r["id"] in best])
    a0 = dc.read_jsonl(T / f"{name}_a0.jsonl")
    summary[name] = {"n": len(recs), "first_try_pass": sum(ok(r) for r in a0), "final": len(best),
                     **{k: sum(r[k] for r in a0) for k in RULE_KEYS if k != "pass"}}
meta["train_data"] = summary
print(json.dumps(summary, ensure_ascii=False, indent=1))

## 평가 (iii) 선생: 평가 입력 934개, greedy, 재생성 없음

In [ ]:
GREEDY = {"temperature": 0.0, "max_tokens": dc.MAX_NEW_TOKENS}
evals = dc.eval_inputs()
E = dc.DRIVE / "eval"
generate(evals, E / "teacher.jsonl", 0, GREEDY)

In [ ]:
# 평균 생성 시간: batch 1, 평가 입력에서 고정 seed로 뽑은 50개
timing_path = E / "teacher_timing.json"
if not timing_path.exists():
    rows = []
    for r in dc.timing_subset(evals):
        t0 = time.time()
        o = llm.chat([prompt.messages(r["input"])], [SamplingParams(**GREEDY)], chat_template_kwargs=CT, use_tqdm=False)
        rows.append({"id": r["id"], "sec": round(time.time() - t0, 2), "new_tokens": len(o[0].outputs[0].token_ids)})
    used = !nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits
    timing = {"mean_sec": round(sum(x["sec"] for x in rows) / len(rows), 2), "n": len(rows),
              "nvidia_smi_used_gb": round(int(used[0]) / 1024, 2), "note": "vLLM은 gpu_memory_utilization만큼 미리 잡으므로 학생 VRAM과 비교하지 않는다",
              "gpu": torch.cuda.get_device_name(0), "rows": rows}
    timing_path.write_text(json.dumps(timing, ensure_ascii=False, indent=1))
print(json.loads(timing_path.read_text())["mean_sec"])
(dc.DRIVE / "teacher" / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))

## repo로 가져올 파일
- `teacher/pilot_*.jsonl`, `teacher/train_final.jsonl`, `teacher/val_final.jsonl`, `teacher/train_a*.jsonl`, `teacher/val_a*.jsonl`, `teacher/meta.json` → `distill/teacher/`
- `eval/teacher.jsonl`, `eval/teacher_timing.json` → `distill/eval/`